# Week 8 (starter): Integrated LLM System (capstone)

A small skeleton that integrates three techniques you already built: retrieval (Week 5), a tool with routing (Week 4), and a grounded prompt (Week 3). Orchestration runs locally with no key; synthesis and the LLM-as-judge are guarded. **Reuse your own components**; this is a starting frame, not the finished system. Cells marked **TODO (you)** are yours.

In [ ]:
import os, re, ast, operator, pathlib, numpy as np
os.environ['HF_HOME']=str((pathlib.Path('.')/'.hf_cache').resolve())
from sentence_transformers import SentenceTransformer
import faiss
embedder=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# technique 1: retrieval. TODO (you): swap in your real corpus / pipeline.
DOC='''Orbit listens on port 7700. Storage is an LSM-tree. Replication factor defaults to 3.\nMax value size is 16 MB. A batch holds at most 1000 operations. Clients use API tokens.'''
chunks=[s.strip() for s in DOC.split('.') if s.strip()]
cvec=embedder.encode(chunks, normalize_embeddings=True).astype('float32')
index=faiss.IndexFlatIP(cvec.shape[1]); index.add(cvec)
def retrieve(q,k=3):
    qe=embedder.encode([q],normalize_embeddings=True).astype('float32'); return [chunks[i] for i in index.search(qe,k)[1][0]]

# technique 2: a tool + router
_OPS={ast.Add:operator.add,ast.Sub:operator.sub,ast.Mult:operator.mul,ast.Div:operator.truediv,ast.Pow:operator.pow}
def _ev(n):
    if isinstance(n,ast.Constant): return n.value
    if isinstance(n,ast.BinOp): return _OPS[type(n.op)](_ev(n.left),_ev(n.right))
    raise ValueError('arithmetic only')
def calculator(expr): return _ev(ast.parse(expr,mode='eval').body)
def route(q): return 'tool' if any(w in q.lower() for w in ['how many','fit','calculate','per ']) else 'rag'
print('components ready: retrieval, calculator tool, router')

In [ ]:
# technique 3: grounded prompt + synthesis (guarded)
def gemini_chat(messages, model='gemini-2.5-flash'):
    if not os.environ.get('GEMINI_API_KEY'): return None
    from openai import OpenAI
    c=OpenAI(api_key=os.environ['GEMINI_API_KEY'], base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
    return c.chat.completions.create(model=model, messages=messages).choices[0].message.content

def assistant(q, tool_expr=None):
    path=route(q); ctx=retrieve(q); tool=calculator(tool_expr) if (path=='tool' and tool_expr) else None
    prompt='Answer from the context and tool result only.\nContext:\n- '+'\n- '.join(ctx)+(f'\nTool result: {tool}' if tool is not None else '')+f'\nQuestion: {q}\nAnswer:'
    out=gemini_chat([{'role':'user','content':prompt}])
    return {'route':path,'tool':tool,'answer': out if out is not None else '[API-BLOCKED] grounded prompt built; set GEMINI_API_KEY to synthesize'}

print(assistant('what port does orbit use'))
print(assistant('how many 16MB values fit in 1024MB', tool_expr='1024/16'))

## Part 2: Evaluate (model-free metrics + guarded judge)
Routing accuracy and answer-grounding are measurable without the model. The LLM-as-judge pass is guarded.

In [ ]:
evalset=[{'q':'what port does orbit use','route':'rag','fact':'7700','expr':None},
  {'q':'which storage engine','route':'rag','fact':'LSM','expr':None},
  {'q':'how many 16MB values fit in 1024MB','route':'tool','fact':None,'expr':'1024/16'}]
route_ok=sum(route(e['q'])==e['route'] for e in evalset)/len(evalset)
ground=sum((e['fact'].lower() in ' '.join(retrieve(e['q'])).lower()) if e['route']=='rag' else (calculator(e['expr']) is not None) for e in evalset)/len(evalset)
print(f'routing accuracy {route_ok:.0%}, answer-grounding {ground:.0%}')
if not os.environ.get('GEMINI_API_KEY'):
    print('[API-BLOCKED] LLM-as-judge needs a key; judge 10 to 20 cases when enabled.')
# TODO (you): add the LLM-as-judge pass and report its scores.

## Part 3 and 4: failure, submit, curate
**TODO (you):** show one failure of the integrated system, trace it to a component, and mitigate. Write a README documenting the architecture and technique choices, then curate the repo as your portfolio piece. Rubric: integration by reuse (30), evaluation incl judge (25), failure traced (15), README (20), portfolio curation (10).